# Lymphocyte subclustering - PCA, UMAP, Leiden (res 0.1)

Recomputes PCA within the lymphocyte subset rather than reusing the global scVI
latent, then UMAP and Leiden at resolution 0.1, followed by feature plots and violin
plots for the `cd8_subclustering_long` genes.

Note: PCA on the subset drops the scVI batch correction. The batch panel in the UMAP
figure is the check - if the clusters track batch rather than markers, that is the
batch effect, not biology.

## 1. Load the object

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt

sc.settings.verbosity = 1

H5AD_PATH = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/"
    "AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/"
    "RESULTS/20240627__192310__KAECH_AD_GBM_240627/08_Plaque_Proximity_Analysis/"
    "adata_combined_with_alphashape_plaque_edge_15_20_30_40um.h5ad"
)

MARKER_XLSX_PATH = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/"
    "Gene List of sub-clustering/BRAIN_CLUSTERING_GUIDE_IRENE_Vs.xlsx"
)
MARKER_SHEET = "cd8_subclustering_long"

OUTPUT_DIR = Path(
    "/Users/valishashah/Library/CloudStorage/Box-Box/Kaech Lab Folder/Valisha/"
    "AD Serial Infection Project  (Irene & Brian W)/Spatial Transcriptomics 20260518/"
    "RESULTS/20240627__192310__KAECH_AD_GBM_240627/10_Lymphocyte_Subclustering"
)
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

adata = ad.read_h5ad(H5AD_PATH)

# X is already normalised + log1p (uns["log1p"] is set and the values are not
# integers); layers["counts"] holds the raw counts.
print(adata)
print()
print("X is log1p-normalised:", "log1p" in adata.uns)
print("raw counts layer:", list(adata.layers))
print("latent space:", adata.obsm["X_scVI"].shape)

## 2. Phenotype signatures from the Excel sheet

Handles genes listed twice, genes the panel does not carry, and the Trm genes the
sheet marks as *down* rather than up.

In [ ]:
# Phenotype signatures from the Excel sheet. The sheet is long format:
# one row per gene, with the phenotype it belongs to.
MINIMUM_GENES_PER_SIGNATURE = 2

marker_sheet = pd.read_excel(MARKER_XLSX_PATH, sheet_name=MARKER_SHEET)
marker_sheet = marker_sheet.rename(columns={"Unnamed: 2": "note"})
marker_sheet["Gene"] = marker_sheet["Gene"].astype(str).str.strip()
marker_sheet["Phenotype"] = marker_sheet["Phenotype"].astype(str).str.strip()

# The sheet marks a few Trm genes as "down"; those must not be scored as if
# high expression supported the phenotype.
down_genes = marker_sheet["note"].astype(str).str.lower().eq("down")
print("genes flagged 'down' in the sheet (excluded from scoring):")
print(marker_sheet.loc[down_genes, ["Gene", "Phenotype"]].to_string(index=False))
marker_sheet = marker_sheet.loc[~down_genes]

panel_genes = set(adata.var_names)
marker_sheet["in_panel"] = marker_sheet["Gene"].isin(panel_genes)

# Deduplicate: some genes are listed twice within a phenotype.
signature_genes = {}
for phenotype, group in marker_sheet.groupby("Phenotype", sort=False):
    present = sorted(set(group.loc[group["in_panel"], "Gene"]))
    signature_genes[phenotype] = present

coverage = pd.DataFrame(
    {
        "genes_in_sheet": marker_sheet.groupby("Phenotype", sort=False)["Gene"].nunique(),
        "genes_in_panel": {k: len(v) for k, v in signature_genes.items()},
    }
)
coverage["genes"] = [", ".join(signature_genes[p]) for p in coverage.index]
coverage["scored"] = coverage["genes_in_panel"] >= MINIMUM_GENES_PER_SIGNATURE

SCORED_SIGNATURES = {
    phenotype: genes
    for phenotype, genes in signature_genes.items()
    if len(genes) >= MINIMUM_GENES_PER_SIGNATURE
}

# The sheet is a CD8 sheet: it has no B-cell, myeloid or neuronal signature, so
# on its own it would give a B-cell subcluster the nearest CD8 label available
# ("Naive like", via Sell / Ccr7 / Cxcr5). These lineage checks are added so a
# subcluster that is not a CD8 state shows up as one instead of being mislabelled.
# Edit freely - they are not from the workbook.
LINEAGE_CHECK_SIGNATURES = {
    "CHECK: T cell lineage": ["Cd3e", "Cd8a", "Cd8b1", "Trbc1", "Cd2", "Thy1"],
    "CHECK: B cell lineage": ["Cd19", "Ms4a1", "Ighd", "Ighm", "Cd74", "H2-Ab1"],
    "CHECK: myeloid lineage": ["Csf1r", "Itgam", "Aif1", "C1qa", "P2ry12"],
    "CHECK: neuronal (ambient/doublet)": [
        "Chgb", "Tubb3", "Dclk1", "Ncam1", "Gpm6b", "Rbfox3",
    ],
}
for check_name, check_genes in LINEAGE_CHECK_SIGNATURES.items():
    present = [gene for gene in check_genes if gene in panel_genes]
    if len(present) >= MINIMUM_GENES_PER_SIGNATURE:
        SCORED_SIGNATURES[check_name] = present

# Every sheet gene that exists in the panel, for the dotplot later.
ALL_MARKER_GENES = sorted(set(marker_sheet.loc[marker_sheet["in_panel"], "Gene"]))

print()
print(f"Panel: {adata.n_vars} genes. Sheet: {marker_sheet['Gene'].nunique()} unique genes, "
      f"{len(ALL_MARKER_GENES)} of them on the panel.")
print(f"Signatures with >= {MINIMUM_GENES_PER_SIGNATURE} panel genes: {len(SCORED_SIGNATURES)} "
      f"of {len(signature_genes)}")
display(coverage.sort_values("genes_in_panel", ascending=False))

## 3. Subset the lymphocytes

In [ ]:
CELLTYPE_KEY = "manual_celltype"
LYMPHOCYTE_LABEL = "Lymphocytes"

lymphocytes = adata[adata.obs[CELLTYPE_KEY] == LYMPHOCYTE_LABEL].copy()

print("Lymphocytes:", lymphocytes.n_obs)
print()
print(lymphocytes.obs["batch"].value_counts().to_string())
print()
print("Counts per cell (raw):")
print(lymphocytes.obs["transcript_counts"].describe().round(1).to_string())
print()
print("Genes detected per cell:")
print(lymphocytes.obs["n_genes_by_counts"].describe().round(1).to_string())

# Sanity check: do these cells actually look like T/NK cells on the panel?
check_genes = [g for g in ["Cd8a", "Cd4", "Cd3e", "Cd19", "Klrb1", "Ptprc"]
               if g in adata.var_names]
detected_fraction = pd.Series(
    {
        gene: float((lymphocytes[:, gene].X.toarray() > 0).mean())
        for gene in check_genes
    },
    name="fraction_of_lymphocytes_expressing",
)
print()
print("Lineage gene detection within the subset:")
display(detected_fraction.round(3).to_frame())

## 4. PCA -> UMAP -> Leiden

`X` is log1p-normalised on the way in. PCA needs scaled data, but the feature and
violin plots must show log-normalised expression, so X is copied to a layer,
scaled for the PCA, then restored.

In [ ]:
# Fresh PCA -> UMAP -> Leiden on the lymphocytes only.
LEIDEN_RESOLUTION = 0.1
LEIDEN_KEY = "lymphocyte_leiden"
N_PCS = 30
N_NEIGHBORS = 10

# X is log1p-normalised. Keep a copy before scaling: PCA wants scaled data,
# but the feature and violin plots must show log-normalised expression, not
# z-scores, or the colour bars and violin axes are meaningless.
lymphocytes.layers["lognorm"] = lymphocytes.X.copy()

# All 480 panel genes are used. The panel is already a targeted gene set, so
# HVG selection on 429 cells mostly just discards signal; set USE_HVG = True
# if you want the standard step anyway.
USE_HVG = False
if USE_HVG:
    sc.pp.highly_variable_genes(lymphocytes, n_top_genes=300)
    print("HVGs:", int(lymphocytes.var["highly_variable"].sum()))

sc.pp.scale(lymphocytes, max_value=10)
sc.tl.pca(lymphocytes, n_comps=N_PCS, svd_solver="arpack", random_state=0)

# Put log-normalised values back for every downstream plot.
lymphocytes.X = lymphocytes.layers["lognorm"].copy()

variance_ratio = lymphocytes.uns["pca"]["variance_ratio"]
print(f"PCA on {lymphocytes.n_obs} cells x {lymphocytes.n_vars} genes")
print("variance explained by first 10 PCs:", np.round(variance_ratio[:10], 4))
print("cumulative at 30 PCs:", round(float(variance_ratio.sum()), 3))

sc.pp.neighbors(lymphocytes, n_pcs=N_PCS, n_neighbors=N_NEIGHBORS)
sc.tl.umap(lymphocytes, min_dist=0.3, random_state=0)
sc.tl.leiden(
    lymphocytes, resolution=LEIDEN_RESOLUTION, key_added=LEIDEN_KEY,
    flavor="igraph", n_iterations=2, directed=False, random_state=0,
)

cluster_sizes = lymphocytes.obs[LEIDEN_KEY].value_counts().sort_index()
print()
print(f"Leiden resolution {LEIDEN_RESOLUTION}: "
      f"{lymphocytes.obs[LEIDEN_KEY].nunique()} subclusters")
print(cluster_sizes.to_string())

cluster_composition = pd.crosstab(lymphocytes.obs[LEIDEN_KEY], lymphocytes.obs["batch"])
cluster_composition["total"] = cluster_composition.sum(axis=1)
cluster_composition["percent_infected"] = (
    100 * cluster_composition.get("AD_inf", 0) / cluster_composition["total"]
)
display(cluster_composition.round(1))

CLUSTER_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                  "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
lymphocytes.uns[f"{LEIDEN_KEY}_colors"] = CLUSTER_COLORS[
    : lymphocytes.obs[LEIDEN_KEY].nunique()
]

# Elbow plus the clusters and batch on the same UMAP. PCA on the subset drops
# the scVI batch correction, so check the batch panel: if the clusters line up
# with batch rather than with markers, that is the batch effect, not biology.
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(range(1, len(variance_ratio) + 1), variance_ratio, marker="o",
        markersize=4, color="#2a78d6")
ax.set_xlabel("principal component")
ax.set_ylabel("variance ratio")
ax.set_title(f"PCA elbow ({N_PCS} PCs computed)", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "pca_elbow.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

fig = sc.pl.umap(
    lymphocytes, color=[LEIDEN_KEY, "batch", "transcript_counts"], ncols=3,
    wspace=0.3, frameon=False, size=60, return_fig=True,
)
fig.savefig(FIGURE_DIR / "pca_umap_clusters.png", dpi=300,
            bbox_inches="tight", facecolor="white")
plt.show()

## 5. What separates the subclusters

Read this before the feature plots - it tells you what the clusters actually are.

In [ ]:
# What separates the subclusters, so the feature and violin plots can be read.
sc.tl.rank_genes_groups(
    lymphocytes, groupby=LEIDEN_KEY, method="wilcoxon", tie_correct=True,
)

marker_table = sc.get.rank_genes_groups_df(lymphocytes, group=None)
marker_table = marker_table.rename(columns={"group": "subcluster"})
significant_markers = marker_table.loc[
    (marker_table["pvals_adj"] < 0.05) & (marker_table["logfoldchanges"] > 0.5)
]

print("Significant up-markers per subcluster (padj < 0.05, logFC > 0.5):")
print(significant_markers["subcluster"].value_counts().sort_index().to_string())
print()
for cluster_name in lymphocytes.obs[LEIDEN_KEY].cat.categories:
    top = significant_markers.loc[significant_markers["subcluster"] == cluster_name].head(12)
    print(f"cluster {cluster_name}: " + ", ".join(top["names"]))

print()
print("Detection depth per subcluster:")
display(
    lymphocytes.obs.groupby(LEIDEN_KEY, observed=True)[
        ["transcript_counts", "n_genes_by_counts", "cell_area"]
    ].median().round(1)
)

print("Top 10 per subcluster:")
display(
    significant_markers.groupby("subcluster", observed=True)
    .head(10)[["subcluster", "names", "logfoldchanges", "pvals_adj"]]
    .round(4)
)

## 6. Feature plots

One figure per phenotype. Two adjustments over the scanpy default: genes detected in
fewer than 5 cells are dropped rather than plotted as noise, and the colour ramp
starts at light gray instead of white so the non-expressing cells stay visible as
the UMAP outline.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

FEATURE_DIR = FIGURE_DIR / "feature_plots"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# A gene detected in a handful of cells produces a panel of noise, so drop
# those rather than plotting them. Set to 0 to keep everything.
MINIMUM_CELLS_DETECTED = 5

expression_matrix = lymphocytes[:, ALL_MARKER_GENES].X
expression_matrix = (
    expression_matrix.toarray() if hasattr(expression_matrix, "toarray") else expression_matrix
)
cells_detected = pd.Series(
    (expression_matrix > 0).sum(axis=0), index=ALL_MARKER_GENES, name="cells_detected"
).sort_values(ascending=False)

PLOTTED_GENES = cells_detected.loc[cells_detected >= MINIMUM_CELLS_DETECTED].index.tolist()
dropped_genes = cells_detected.loc[cells_detected < MINIMUM_CELLS_DETECTED]

print(f"{len(PLOTTED_GENES)} of {len(ALL_MARKER_GENES)} sheet genes detected in "
      f">= {MINIMUM_CELLS_DETECTED} of the {lymphocytes.n_obs} lymphocytes.")
print()
print("Not plotted (too sparse):")
print(dropped_genes.to_string())

# scanpy's default puts zero at white, which erases the UMAP outline and leaves
# only the expressing cells visible. Starting the ramp at light gray keeps the
# non-expressing cells on screen as context.
EXPRESSION_CMAP = LinearSegmentedColormap.from_list(
    "gray_to_blue", ["#e6e5e1", "#9ec5f4", "#2a78d6", "#104281"]
)


def safe_filename(name):
    for character in " /()":
        name = name.replace(character, "_")
    return name.replace("__", "_").strip("_")


# One figure per phenotype, so each set can be read on its own.
for phenotype, genes in signature_genes.items():
    plot_genes = [gene for gene in genes if gene in PLOTTED_GENES]
    if not plot_genes:
        print(f"skipped feature plot for {phenotype!r}: no gene detected often enough")
        continue

    fig = sc.pl.umap(
        lymphocytes, color=plot_genes, ncols=4, frameon=False, size=55,
        cmap=EXPRESSION_CMAP, vmin=0, vmax="p99", wspace=0.25, return_fig=True,
    )
    fig.suptitle(f"{phenotype}  ({len(plot_genes)} genes)", y=1.01,
                 fontsize=13, x=0.02, ha="left")
    fig.savefig(FEATURE_DIR / f"umap_{safe_filename(phenotype)}.png", dpi=200,
                bbox_inches="tight", facecolor="white")
    plt.show()

print("Feature plots written to:", FEATURE_DIR)

# And one combined grid of every plotted gene, as an overview.
fig = sc.pl.umap(
    lymphocytes, color=PLOTTED_GENES, ncols=6, frameon=False, size=40,
    cmap=EXPRESSION_CMAP, vmin=0, vmax="p99", wspace=0.25, return_fig=True,
)
fig.savefig(FIGURE_DIR / "umap_all_sheet_genes.png", dpi=150,
            bbox_inches="tight", facecolor="white")
plt.show()

## 7. Violin plots by Leiden cluster

`sc.pl.violin` puts every gene in one row, which is unreadable past about four genes,
so the grid is built directly. The jittered points are kept deliberately: these
counts are sparse enough that a violin over mostly zeros is easy to over-read.

In [ ]:
VIOLIN_DIR = FIGURE_DIR / "violin_plots"
VIOLIN_DIR.mkdir(parents=True, exist_ok=True)

cluster_names = lymphocytes.obs[LEIDEN_KEY].cat.categories.tolist()
cluster_colors = list(lymphocytes.uns[f"{LEIDEN_KEY}_colors"])
jitter_rng = np.random.default_rng(0)


def plot_violin_grid(genes, title, output_path, ncols=4):
    """Violins per Leiden cluster, wrapped into a grid.

    sc.pl.violin puts every gene in a single row, which is unreadable past
    about four genes, so the grid is built directly. Expression is the
    log-normalised values in X. The jittered points matter here: these counts
    are sparse enough that a violin over mostly zeros is easy to over-read.
    """
    genes = [gene for gene in genes if gene in lymphocytes.var_names]
    if not genes:
        return None

    nrows = int(np.ceil(len(genes) / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(3.3 * ncols, 2.7 * nrows), squeeze=False
    )

    cluster_masks = [
        (lymphocytes.obs[LEIDEN_KEY] == name).to_numpy() for name in cluster_names
    ]

    for index, gene in enumerate(genes):
        ax = axes[index // ncols][index % ncols]
        values = lymphocytes[:, gene].X
        values = np.asarray(values.toarray() if hasattr(values, "toarray") else values).ravel()
        per_cluster = [values[mask] for mask in cluster_masks]

        for position, (cluster_values, color) in enumerate(
            zip(per_cluster, cluster_colors), start=1
        ):
            # violinplot needs some spread; an all-zero cluster gets points only.
            if cluster_values.std() > 0:
                parts = ax.violinplot(
                    cluster_values, positions=[position], widths=0.75,
                    showextrema=False, showmedians=True,
                )
                for body in parts["bodies"]:
                    body.set_facecolor(color)
                    body.set_alpha(0.65)
                    body.set_edgecolor("none")
                parts["cmedians"].set_color("#0b0b0b")
                parts["cmedians"].set_linewidth(1.2)

            jitter = jitter_rng.uniform(-0.11, 0.11, len(cluster_values))
            ax.scatter(
                position + jitter, cluster_values, s=2.5, color="#2b2b2b",
                alpha=0.30, linewidths=0, zorder=3,
            )

        ax.set_xticks(range(1, len(cluster_names) + 1))
        ax.set_xticklabels(cluster_names, fontsize=9)
        ax.set_title(gene, fontsize=10, color="#0b0b0b")
        ax.set_ylabel("log-norm expression", fontsize=8, color="#52514e")
        ax.tick_params(axis="both", labelsize=8, colors="#52514e")
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", color="#e3e2dd", linewidth=0.7)
        ax.set_axisbelow(True)

    for index in range(len(genes), nrows * ncols):
        axes[index // ncols][index % ncols].axis("off")

    fig.suptitle(title, y=1.0, fontsize=13, x=0.01, ha="left")
    fig.supxlabel(LEIDEN_KEY, fontsize=9, color="#52514e")
    plt.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()
    return fig


# One grid per phenotype, using the same detection filter as the feature plots.
for phenotype, genes in signature_genes.items():
    plot_genes = [gene for gene in genes if gene in PLOTTED_GENES]
    if not plot_genes:
        continue
    plot_violin_grid(
        plot_genes,
        f"{phenotype}  -  by {LEIDEN_KEY} (resolution {LEIDEN_RESOLUTION})",
        VIOLIN_DIR / f"violin_{safe_filename(phenotype)}.png",
    )

print("Violin plots written to:", VIOLIN_DIR)

# The data-driven markers, independent of the sheet.
top_de_genes = (
    significant_markers.groupby("subcluster", observed=True).head(6)["names"].unique().tolist()
)
plot_violin_grid(
    top_de_genes,
    f"Top DE markers  -  by {LEIDEN_KEY} (resolution {LEIDEN_RESOLUTION})",
    FIGURE_DIR / "violin_top_de_markers.png",
)

# Compact overview of everything at once.
stacked_groups = {
    phenotype: [gene for gene in genes if gene in PLOTTED_GENES]
    for phenotype, genes in SCORED_SIGNATURES.items()
}
stacked_groups = {name: genes for name, genes in stacked_groups.items() if genes}

sc.pl.stacked_violin(
    lymphocytes, var_names=stacked_groups, groupby=LEIDEN_KEY,
    standard_scale="var", cmap="Blues", show=False,
)
plt.savefig(FIGURE_DIR / "stacked_violin_all_signatures.png", dpi=200,
            bbox_inches="tight", facecolor="white")
plt.show()

## 8. Save

In [ ]:
SUBSET_H5AD_PATH = OUTPUT_DIR / f"lymphocytes_pca_leiden_res{LEIDEN_RESOLUTION}.h5ad"
RESULTS_XLSX_PATH = OUTPUT_DIR / f"lymphocyte_pca_res{LEIDEN_RESOLUTION}_results.xlsx"

lymphocytes.uns["lymphocyte_subclustering"] = {
    "source_h5ad": str(H5AD_PATH),
    "marker_workbook": str(MARKER_XLSX_PATH),
    "marker_sheet": MARKER_SHEET,
    "celltype_key": CELLTYPE_KEY,
    "celltype_label": LYMPHOCYTE_LABEL,
    "representation": f"PCA recomputed on the subset ({N_PCS} PCs)",
    "used_highly_variable_genes": bool(USE_HVG),
    "n_genes_used": int(lymphocytes.n_vars),
    "n_neighbors": int(N_NEIGHBORS),
    "leiden_resolution": float(LEIDEN_RESOLUTION),
    "leiden_flavor": "igraph",
    "n_subclusters": int(lymphocytes.obs[LEIDEN_KEY].nunique()),
    "n_cells": int(lymphocytes.n_obs),
    "expression_in_X": "log1p-normalised (restored after scaling for PCA)",
    "created": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
}

lymphocytes.write_h5ad(SUBSET_H5AD_PATH, compression="gzip")
print("Wrote", SUBSET_H5AD_PATH.name,
      round(SUBSET_H5AD_PATH.stat().st_size / 1e6, 1), "MB")

# Carry the labels onto the combined object. Positional, because obs_names can
# repeat between the infected and mock sections.
lymphocyte_positions = np.flatnonzero(
    (adata.obs[CELLTYPE_KEY] == LYMPHOCYTE_LABEL).to_numpy()
)
assert len(lymphocyte_positions) == lymphocytes.n_obs
assert (
    adata.obs_names[lymphocyte_positions] == lymphocytes.obs_names
).all(), "row order mismatch"

label_values = np.full(adata.n_obs, None, dtype=object)
label_values[lymphocyte_positions] = lymphocytes.obs[LEIDEN_KEY].astype(object).to_numpy()
adata.obs[LEIDEN_KEY] = pd.Categorical(label_values, categories=cluster_names)
print()
print(adata.obs[LEIDEN_KEY].value_counts(dropna=False).to_string())

mean_expression = pd.DataFrame(
    {
        name: np.asarray(
            lymphocytes[mask, PLOTTED_GENES].X.mean(axis=0)
        ).ravel()
        for name, mask in zip(
            cluster_names,
            [(lymphocytes.obs[LEIDEN_KEY] == n).to_numpy() for n in cluster_names],
        )
    },
    index=PLOTTED_GENES,
)
percent_detected = pd.DataFrame(
    {
        name: 100 * np.asarray(
            (lymphocytes[mask, PLOTTED_GENES].X > 0).mean(axis=0)
        ).ravel()
        for name, mask in zip(
            cluster_names,
            [(lymphocytes.obs[LEIDEN_KEY] == n).to_numpy() for n in cluster_names],
        )
    },
    index=PLOTTED_GENES,
)

with pd.ExcelWriter(RESULTS_XLSX_PATH, engine="openpyxl") as writer:
    cluster_composition.to_excel(writer, sheet_name="cluster_composition")
    coverage.to_excel(writer, sheet_name="signature_coverage")
    cells_detected.to_frame().to_excel(writer, sheet_name="gene_detection")
    mean_expression.round(4).to_excel(writer, sheet_name="mean_expression")
    percent_detected.round(2).to_excel(writer, sheet_name="percent_detected")
    marker_table.to_excel(writer, sheet_name="DE_all", index=False)
    significant_markers.to_excel(writer, sheet_name="DE_significant", index=False)
    for cluster_name in cluster_names:
        subset = significant_markers.loc[significant_markers["subcluster"] == cluster_name]
        subset.to_excel(writer, sheet_name=f"cluster_{cluster_name}"[:31], index=False)

print()
print("Wrote", RESULTS_XLSX_PATH.name)
print("Figures in:", FIGURE_DIR)

## Caveats

**Resolution 0.1 gives 2 clusters, and cluster 1 is a mixture.** Its markers are both
B-cell (Ighd, Ms4a1, Cd19, Cd74, H2-Ab1, Ighm) and neuronal (Chgb, Chga, Tubb3,
Dclk1, Cx3cl1) at the lowest detection depth of the two - so it merges genuine B
cells with what looks like ambient RNA or mis-segmentation. Raising
`LEIDEN_RESOLUTION` to about 0.4 separates them.

**Only cluster 0 is a T-cell population** (Cd3e, Cd8a, Cd8b1, Trbc1, Ccl5, Cxcr3,
Cxcr6, Il7r). None of the CD8 states in the sheet are resolved at this resolution.

**PCA explains little variance.** The first 10 PCs cover 27% and all 30 cover 43%,
which is what 429 cells at ~80 genes detected each looks like. Treat the UMAP as a
layout, not as evidence of distinct states.